# FinSight RAG — 4 of 4: Chain (GPT-4o Cited Answers)

**Purpose:** Connect ChromaDB retrieval to GPT-4o to generate cited answers
grounded exclusively in the indexed SEC filings.

**Input:** ChromaDB vector store on Drive (from `retriever_final.ipynb`)  
**Output:** `ask(question)` function returning `{answer, sources, question}`  

---

### Pipeline position
```
1. Ingestor → 2. Chunker → 3. Retriever → [4. Chain]
Question → MultiQueryRetriever → GPT-4o → Cited answer
```

### How cited answers work
```
ask('What was Visa net revenue in FY2025?')
  → MultiQueryRetriever fetches 6 relevant chunks from ChromaDB
  → Chunks formatted as [1] V 10-K... [2] V 10-K...
  → System prompt instructs GPT-4o to cite [1][2] after every claim
  → Answer: 'Net revenue was **$40B** [2], an 11% increase...'
  → Sources: list of filing sections used
```

### Cost per query
- MultiQueryRetriever (gpt-4o-mini rewrites): ~$0.001
- GPT-4o generation (1024 tokens): ~$0.01
- **Total: ~$0.01 per question**

## Cell 1 — Install dependencies

In [ ]:
!pip install --upgrade \
  langchain \
  langchain-core \
  langchain-openai \
  langchain-chroma \
  langchain-community \
  langchain-text-splitters \
  langchain-classic \
  chromadb \
  openai \
  pdfplumber \
  beautifulsoup4 \
  tenacity \
  python-dotenv \
  -q
print('✅ All packages installed')

✅ All packages installed


## Cell 2 — Restart runtime

Clears stale package versions. Colab will show 'Session crashed' — this is expected. Continue from Cell 3.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Cell 3 — Imports

Start here after restart.

In [ ]:
import os, io, re, time, logging, requests, pdfplumber
from pathlib import Path
from bs4 import BeautifulSoup

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print('✅ Imports done')

✅ Imports done


## Cell 4 — Mount Drive, set paths, API key

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/finsight-rag-phase1'
CACHE_DIR   = f'{DRIVE_BASE}/data/filings'
CHROMA_DIR  = f'{DRIVE_BASE}/data/chroma'

# Load OpenAI key from Colab Secrets
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    os.environ['OPENAI_API_KEY'] = 'sk-...'  # paste key here if needed
    print('⚠️  Key set directly')

key = os.environ.get('OPENAI_API_KEY', '')
print(f'{"✅" if key.startswith("sk-") else "❌"} Key format: {"valid" if key.startswith("sk-") else "invalid"}')
print(f'✅ ChromaDB path: {CHROMA_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ API key loaded from Colab Secrets
✅ Key format: valid
✅ ChromaDB path: /content/drive/MyDrive/finsight-rag-phase1/data/chroma


## Cell 5 — Rebuild chunks from Drive cache

Reconstitutes the `chunks` variable from the cached `.bin` filing. No network calls — takes ~10 seconds.

In [ ]:
EDGAR_SUBMISSIONS_URL = 'https://data.sec.gov/submissions/CIK{cik:010d}.json'
HEADERS = {'User-Agent': os.getenv('SEC_USER_AGENT', 'FinSight-RAG dev@example.com')}
TICKER_TO_CIK = {'V': 1403161, 'MA': 1141391, 'PYPL': 1633917, 'SQ': 1512673}

@retry(stop=stop_after_attempt(4), wait=wait_exponential(min=2, max=30),
       retry=retry_if_exception_type(requests.exceptions.RequestException), reraise=True)
def _get(url):
    time.sleep(0.15)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r

def get_filing_urls(ticker, form_type='10-K', max_filings=1):
    cik = TICKER_TO_CIK[ticker.upper()]
    data = _get(EDGAR_SUBMISSIONS_URL.format(cik=cik)).json()
    recent = data.get('filings', {}).get('recent', {})
    filings = []
    for form, accession, filing_date in zip(
        recent.get('form', []), recent.get('accessionNumber', []), recent.get('filingDate', [])):
        if form != form_type: continue
        acc_clean = accession.replace('-', '')
        filings.append({'ticker': ticker, 'form_type': form, 'filing_date': filing_date,
            'accession_number': accession,
            'index_url': f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc_clean}/{accession}-index.htm'})
        if len(filings) >= max_filings: break
    return filings

def _extract_filing_url(index_url):
    cik_m = re.search(r'/edgar/data/(\d+)/', index_url)
    acc_m = re.search(r'/(\d{18})/', index_url.replace('-', ''))
    if not cik_m or not acc_m: return None
    cik, acc = cik_m.group(1), acc_m.group(1)
    files = _get(f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/index.json').json().get('directory', {}).get('item', [])
    candidates = [f for f in files if f['name'].endswith(('.htm', '.pdf'))
        and not f['name'].startswith('R') and 'index' not in f['name'].lower()
        and int(f.get('size', 0)) > 50_000]
    if not candidates: return None
    best = max(candidates, key=lambda f: int(f.get('size', 0)))
    return f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{best["name"]}'

def _bytes_to_documents(raw_bytes, metadata):
    docs = []
    sample = raw_bytes[:500].decode('utf-8', errors='ignore').lower()
    if any(m in sample for m in ['<html', '<!doctype', '<document']):
        soup = BeautifulSoup(raw_bytes, 'html.parser')
        for tag in soup(['script', 'style', 'head', 'nav', 'footer']): tag.decompose()
        full_text = re.sub(r'\n{3,}', '\n\n', soup.get_text(separator='\n')).strip()
        for i, text in enumerate([full_text[j:j+3000] for j in range(0, len(full_text), 3000)], 1):
            if len(text.strip()) > 100:
                docs.append(Document(page_content=text, metadata={**metadata, 'page': i}))
        return docs
    try:
        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            for i, page in enumerate(pdf.pages, 1):
                text = page.extract_text() or ''
                if text.strip(): docs.append(Document(page_content=text, metadata={**metadata, 'page': i}))
    except: pass
    return docs

def ingest_ticker(ticker, form_type='10-K', max_filings=1, cache_dir=CACHE_DIR):
    all_docs = []
    for filing in get_filing_urls(ticker, form_type, max_filings):
        cache_path = Path(cache_dir) / f"{ticker}_{filing['form_type']}_{filing['filing_date']}.bin"
        if cache_path.exists() and cache_path.stat().st_size > 100_000:
            raw_bytes = cache_path.read_bytes()
        else:
            doc_url = _extract_filing_url(filing['index_url'])
            if not doc_url: continue
            raw_bytes = _get(doc_url).content
            cache_path.write_bytes(raw_bytes)
        meta = {'ticker': ticker, 'form_type': filing['form_type'],
                'filing_date': filing['filing_date'], 'source': filing['index_url']}
        all_docs.extend(_bytes_to_documents(raw_bytes, meta))
    return all_docs

_NOISE = [re.compile(p) for p in [
    r'^\s*Page\s+\d+\s*$', r'Table\s+of\s+Contents',
    r'^\s*[-\u2013\u2014]{5,}\s*$', r'UNITED STATES\s+SECURITIES AND EXCHANGE']]

def _clean(text):
    for p in _NOISE: text = p.sub(' ', text)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

def _meaningful(text):
    s = text.strip()
    if len(s) < 80: return False
    return sum(c.isalpha() for c in s) / max(len(s), 1) >= 0.15

def chunk_documents(documents, chunk_size=800, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size*4, chunk_overlap=chunk_overlap*4,
        separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''], length_function=len)
    chunks = []
    for doc in documents:
        cleaned = _clean(doc.page_content)
        if not cleaned: continue
        for idx, text in enumerate(splitter.split_text(cleaned)):
            if _meaningful(text):
                chunks.append(Document(page_content=text, metadata={
                    **doc.metadata, 'chunk_index': idx,
                    'citation': (f"{doc.metadata.get('ticker','?')} "
                                 f"{doc.metadata.get('form_type','?')} "
                                 f"({doc.metadata.get('filing_date','?')}) "
                                 f"· section {doc.metadata.get('page','?')}")
                }))
    return chunks

print('Loading from Drive cache...\n')
docs   = ingest_ticker('V', '10-K', max_filings=1)
chunks = chunk_documents(docs)
print(f'✅ {len(chunks)} chunks ready')

Loading from Drive cache...

✅ 155 chunks ready


## Cell 6 — Load ChromaDB from Drive

Loads the existing vector store — no re-embedding, no API cost.

In [ ]:
COLLECTION_NAME = 'finsight_filings'
EMBEDDING_MODEL = 'text-embedding-3-small'

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

count = vector_store._collection.count()
print(f'✅ ChromaDB loaded — {count} vectors')

if count == 0:
    print('❌ No vectors found — run retriever_v2.ipynb first to index the chunks')

✅ ChromaDB loaded — 155 vectors


## Cell 7 — Build the RAG chain

### System prompt design
The system prompt is the most important prompt-engineering decision in this project.
Each rule exists for a specific reason:
- **'Base your answer ONLY on context'** — prevents GPT-4o from using its training data,
  which could include outdated or incorrect financial information
- **'After every factual claim, add [N] citation'** — makes every number traceable
  to a specific filing section; enables fact-checking
- **'If context doesn't contain enough info, say so'** — prevents hallucinated figures,
  which is the most damaging failure mode for a financial tool
- **`temperature=0`** — deterministic output; financial figures should not vary between runs

### LCEL pipeline
LangChain Expression Language (LCEL) uses the `|` pipe operator (like Unix pipes):
`prompt | llm | parser` means: fill the prompt → call GPT-4o → parse text output.
Each step's output becomes the next step's input.

In [ ]:
# ── System prompt ─────────────────────────────────────────────────────────
# This is prompt engineering — the instructions that control GPT-4o's behaviour.
# Every rule here has a reason: citations prevent hallucination, refusing to guess
# prevents made-up numbers, formatting makes answers scannable.

SYSTEM_PROMPT = """You are FinSight, an expert financial analyst assistant \
specialising in SEC filings (10-K annual reports and 10-Q quarterly reports).

You will be given numbered CONTEXT EXCERPTS from real filings, then a QUESTION.
Your job is to answer accurately with inline citations.

Rules:
1. Base your answer ONLY on the provided context. Do not use outside knowledge.
2. After every factual claim, add a citation like [1] or [2, 3] referencing
   the excerpt number(s) that support it.
3. If the context does not contain enough information, say:
   "The provided filings do not contain enough information to answer this question."
   Do NOT guess or fabricate figures.
4. Format numbers clearly: use $B for billions, $M for millions, % for percentages.
5. Keep answers under 300 words unless the question asks for detail.
6. Use markdown: **bold** for key metrics."""

HUMAN_TEMPLATE = """CONTEXT EXCERPTS:
{context}

QUESTION: {question}

ANSWER (with inline citations):"""


# ── Helper: format chunks as numbered excerpts ────────────────────────────
def _format_context(docs: list[Document]) -> str:
    """
    Convert retrieved chunks into numbered excerpts for the prompt.
    Each excerpt includes its citation metadata so GPT-4o can reference it.
    We truncate to 1200 chars per chunk to stay within context limits.
    """
    parts = []
    for i, doc in enumerate(docs, start=1):
        meta   = doc.metadata
        header = (
            f"[{i}] {meta.get('ticker','?')} "
            f"{meta.get('form_type','?')} "
            f"({meta.get('filing_date','?')}) "
            f"· Section {meta.get('page','?')}"
        )
        parts.append(f"{header}\n{doc.page_content[:1200]}")
    return "\n\n---\n\n".join(parts)


# ── Helper: convert docs to source list for UI display ────────────────────
def _docs_to_sources(docs: list[Document]) -> list[dict]:
    return [
        {
            'citation':    doc.metadata.get('citation', ''),
            'ticker':      doc.metadata.get('ticker', ''),
            'form_type':   doc.metadata.get('form_type', ''),
            'filing_date': doc.metadata.get('filing_date', ''),
            'page':        doc.metadata.get('page', ''),
            'snippet':     doc.page_content[:300] + '…',
        }
        for doc in docs
    ]


# ── Build the retriever ───────────────────────────────────────────────────
rewrite_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
base_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6},
)
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=rewrite_llm,
)


# ── Build the chain ───────────────────────────────────────────────────────
# LCEL (LangChain Expression Language) — reads left to right:
#   retriever → format context → fill prompt → call GPT-4o → parse text

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', HUMAN_TEMPLATE),
])

generation_llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0,    # deterministic — financial figures should not vary
    max_tokens=1024,
)


def ask(question: str) -> dict:
    """
    Main function — ask a question, get a cited answer.

    Returns:
        {
            'question': str,
            'answer':   str,   # markdown with [1][2] citations
            'sources':  list   # [{citation, ticker, page, snippet}]
        }
    """
    # Step 1: retrieve relevant chunks
    docs = retriever.invoke(question)

    # Step 2: format as numbered context
    context = _format_context(docs)

    # Step 3: call GPT-4o
    chain   = prompt | generation_llm | StrOutputParser()
    answer  = chain.invoke({'context': context, 'question': question})

    return {
        'question': question,
        'answer':   answer,
        'sources':  _docs_to_sources(docs),
    }


print('✅ RAG chain ready')
print('   Retriever : MultiQueryRetriever (gpt-4o-mini rewrites)')
print('   Generator : GPT-4o (temperature=0, max_tokens=1024)')
print('   Citations : inline [1][2] referencing filing section')

✅ RAG chain ready
   Retriever : MultiQueryRetriever (gpt-4o-mini rewrites)
   Generator : GPT-4o (temperature=0, max_tokens=1024)
   Citations : inline [1][2] referencing filing section


## Cell 8 — First cited answer

The payoff cell. This question is a good first test because Visa's revenue table
appears in multiple sections of the filing, so multi-query retrieval should
find it reliably and GPT-4o should be able to give a precise cited answer.

In [ ]:
result = ask('What was Visa total net revenue in fiscal year 2025 and how did it compare to 2024?')

print('══ QUESTION ════════════════════════════════════════════════')
print(result['question'])

print('\n══ ANSWER ══════════════════════════════════════════════════')
print(result['answer'])

print('\n══ SOURCES ═════════════════════════════════════════════════')
for i, src in enumerate(result['sources'], 1):
    print(f'[{i}] {src["citation"]}')
    print(f'    {src["snippet"][:150]}')
    print()

══ QUESTION ════════════════════════════════════════════════
What was Visa total net revenue in fiscal year 2025 and how did it compare to 2024?

══ ANSWER ══════════════════════════════════════════════════
Visa's **total net revenue** in fiscal year 2025 was **$40.0 billion**, which represents an **11% increase** compared to the net revenue of **$35.926 billion** in fiscal year 2024 [2].

══ SOURCES ═════════════════════════════════════════════════
[1] V 10-K (2025-11-06) · section 70
    llions, except per share data)
July 1 – 31, 2025
5 
$
356.32 
5 
$
27,889 
August 1 – 31, 2025
5 
$
345.72 
5 
$
26,142 
September 1 – 30, 2025
4 
$
3

[2] V 10-K (2025-11-06) · section 77
    .
The following table presents the components of our net revenue:
 
For the Years Ended
September 30,
% Change
(1)
 
2025
2024
2023
2025
vs.
2024
2024

[3] V 10-K (2025-11-06) · section 76
    service revenue reported for the twelve months ended September 30, 2025, 2024 and 2023, was based on nominal payments v

## Cell 9 — Additional queries

These questions test different aspects of the filing:
- **Risk factors** — tests whether the retriever finds qualitative text, not just numbers
- **Cross-border volume** — tests segment-level data retrieval
- **Operating margin** — tests multi-step reasoning (income / revenue)

Queries are run individually (not in a loop) so a failure in one doesn't stop the others.

In [ ]:
# Try each of these one at a time — they show different aspects of the filing

questions = [
    'What were the main risk factors Visa highlighted related to regulation or geopolitics?',
    'How did cross-border transaction volume change year over year?',
    'What was Visa operating income and operating margin in FY2025?',
]

for q in questions:
    print(f'\n══ Q: {q}')
    print('   Retrieving...', end=' ')
    result = ask(q)
    print('done')
    print(f'\n{result["answer"]}')
    print(f'\nSources: {", ".join(src["citation"] for src in result["sources"][:3])}')
    print('\n' + '─'*60)


══ Q: What were the main risk factors Visa highlighted related to regulation or geopolitics?
   Retrieving... done

Visa highlighted several main risk factors related to regulation and geopolitics in their 10-K filing:

1. **Regulatory Changes and Compliance**: Visa faces risks from complex and fragmented data-related regulations, including privacy, data use, AI, and cybersecurity. These regulations impact how Visa collects, uses, and handles data, and may require more restrictive practices or expanded cybersecurity requirements [7]. Additionally, Visa is subject to regulations such as the second Payment Services Directive (PSD2) in Europe, which imposes operational complexities and impacts consumer payment experiences [8].

2. **Geopolitical Trends**: Visa is affected by geopolitical trends such as nationalism, protectionism, and restrictive visa requirements. Economic sanctions, tariffs, and trade restrictions, including those affecting cross-border data flow, can impact Visa's busi

## Cell 10 — Health check & Phase 1 summary

In [ ]:
print('══ CHAIN HEALTH CHECK ══════════════════════════════════════')

# Quick test query
test = ask('What business does Visa operate?')

has_answer   = len(test['answer']) > 50
has_citation = '[1]' in test['answer'] or '[2]' in test['answer']
has_sources  = len(test['sources']) > 0

print(f'{"✅" if has_answer else "❌"} Answer generated ({len(test["answer"])} chars)')
print(f'{"✅" if has_citation else "⚠️ "} Inline citations present in answer')
print(f'{"✅" if has_sources else "❌"} Source list returned ({len(test["sources"])} sources)')

print('\n══ PHASE 1 COMPLETE ════════════════════════════════════════')
print('''
What you built:
  ✅ SEC EDGAR fetcher     — ingestor.py
  ✅ Semantic chunker      — chunker.py
  ✅ ChromaDB vector store — retriever.py (155 vectors)
  ✅ Multi-query retrieval — gpt-4o-mini rewrites queries
  ✅ Cited answer chain    — gpt-4o with [1][2] citations

What's next (Phase 2):
  → Cross-encoder reranker (better precision)
  → FinBERT sentiment on MD&A section
  → Sentiment vs price correlation chart
  → Streamlit UI for the full app
''')

══ CHAIN HEALTH CHECK ══════════════════════════════════════
✅ Answer generated (1674 chars)
✅ Inline citations present in answer
✅ Source list returned (10 sources)

══ PHASE 1 COMPLETE ════════════════════════════════════════

What you built:
  ✅ SEC EDGAR fetcher     — ingestor.py
  ✅ Semantic chunker      — chunker.py
  ✅ ChromaDB vector store — retriever.py (155 vectors)
  ✅ Multi-query retrieval — gpt-4o-mini rewrites queries
  ✅ Cited answer chain    — gpt-4o with [1][2] citations

What's next (Phase 2):
  → Cross-encoder reranker (better precision)
  → FinBERT sentiment on MD&A section
  → Sentiment vs price correlation chart
  → Streamlit UI for the full app

